
# Download PitVis-2023 to Google Drive

Downloads **PitVis-2023**
([Das et al., MICCAI PitVis Challenge](https://arxiv.org/abs/2409.01184)) --
25 endoscopic pituitary (skull-base neurosurgery) videos labeled for step
and instrument recognition -- as a single "all files in this version" zip
from UCL's Research Data Repository (Figshare-based):

```
https://rdr.ucl.ac.uk/ndownloader/articles/26531686/versions/2
```

This is the direct file-serving endpoint, not the article page -- the
article page itself sits behind an AWS WAF JavaScript challenge (confirmed
separately: a plain request gets a `202` + `challenge.js`, unsolvable by any
non-browser client). This `ndownloader` URL is a different endpoint and
returned a different failure when tested directly from this notebook's build
environment: a blunt `403 Forbidden` from the load balancer, with none of the
WAF-challenge markers. **That could be an IP-based block specific to the
build environment, or something Colab will also hit -- it wasn't possible to
tell which from here.** Section 1 below reports plainly whether it worked;
if you get a 403 too, the likely next step is trying from a different
network/IP (e.g. a local download in a real browser, then upload to Drive)
rather than debugging the code further.

## What this notebook does

1. Sets up `data/pitvis/` on Drive.
2. Downloads the single bundle zip, resumable (`.part` + `Range`, same as
   the Cholec80 notebook).
3. Checks the response/downloaded file isn't actually an HTML error/challenge
   page saved under the wrong name.
4. Verifies zip integrity (`zipfile.testzip()`, with a size-keyed `.verified`
   marker so repeat runs don't re-scan an unchanged file) before extracting.
5. Extracts.


## 0. Setup: paths and packages

In [ ]:

import sys, subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/endofm-benchmark")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "tqdm"], check=True)
else:
    PROJECT_ROOT = Path(r"E:/endofm-benchmark")

DATA_DIR = PROJECT_ROOT / "data" / "pitvis"
ZIP_PATH = DATA_DIR / "pitvis_all_files_v2.zip"
PART_PATH = ZIP_PATH.with_suffix(ZIP_PATH.suffix + ".part")
VERIFIED_MARKER = DATA_DIR / "pitvis_all_files_v2.zip.verified"
EXTRACT_DIR = DATA_DIR / "raw"

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Data dir:", DATA_DIR)


## 1. Download (resumable) + HTML/challenge-page sanity check

Same resume pattern as the Cholec80 notebook: writes to a `.part` file so a
dropped session loses only the unwritten tail. After downloading, checks
that what actually landed isn't an HTML error/challenge page saved under a
`.zip` name -- a real risk given the WAF sits in front of the article page
for this host.

In [ ]:

import requests
from tqdm.auto import tqdm

PITVIS_URL = "https://rdr.ucl.ac.uk/ndownloader/articles/26531686/versions/2"

BROWSER_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "*/*",
}

def looks_like_html_or_challenge(path, sniff_bytes=512):
    with open(path, "rb") as f:
        head = f.read(sniff_bytes).lstrip().lower()
    return (
        head.startswith(b"<!doctype html") or head.startswith(b"<html")
        or b"<head" in head[:200] or b"awswaf" in head or b"challenge.js" in head
        or b"403 forbidden" in head
    )

def download_pitvis(url, final_path, part_path, chunk_mb=8):
    resume_pos = part_path.stat().st_size if part_path.exists() else 0
    headers = dict(BROWSER_HEADERS)
    if resume_pos:
        headers["Range"] = f"bytes={resume_pos}-"

    with requests.get(url, headers=headers, stream=True, timeout=60) as r:
        print(f"status={r.status_code}  content-type={r.headers.get('Content-Type')!r}  "
              f"content-length={r.headers.get('Content-Length')}")
        if r.status_code == 403:
            raise RuntimeError(
                "403 Forbidden -- likely an IP-based block on this network (confirmed happening "
                "from the build environment too). Try a different network, or download once in a "
                "real browser and upload the result into "
                f"{final_path.parent} manually."
            )
        if resume_pos and r.status_code == 200:
            print("Server doesn't support resuming; restarting from 0.")
            resume_pos = 0
            part_path.unlink()
        r.raise_for_status()

        total = resume_pos + int(r.headers.get("content-length", 0))
        mode = "ab" if resume_pos else "wb"
        with open(part_path, mode) as f, tqdm(
            total=total or None, initial=resume_pos, unit="B", unit_scale=True,
            unit_divisor=1024, desc=final_path.name,
        ) as pbar:
            for chunk in r.iter_content(chunk_size=chunk_mb * 1024 * 1024):
                f.write(chunk)
                pbar.update(len(chunk))

    part_path.rename(final_path)

if ZIP_PATH.exists() and VERIFIED_MARKER.exists() and \
        VERIFIED_MARKER.read_text().strip() == str(ZIP_PATH.stat().st_size):
    print("Already downloaded + verified:", ZIP_PATH)
else:
    download_pitvis(PITVIS_URL, ZIP_PATH, PART_PATH)

    if looks_like_html_or_challenge(ZIP_PATH):
        raise RuntimeError(
            f"{ZIP_PATH} looks like an HTML/challenge/error page, not a real zip -- "
            "the download did not succeed even though the request didn't raise. "
            "Inspect the file's contents directly before retrying."
        )
    print(f"Downloaded: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e9:.3f} GB)")


## 2. Integrity check

`zipfile.testzip()` CRC-checks every member -- catches a truncated/interrupted
download that the size/status-code checks above wouldn't. The `.verified`
marker (keyed to file size) skips re-checking a file that hasn't changed
since last run.

In [ ]:

import zipfile

def check_integrity(zip_path):
    if not zipfile.is_zipfile(zip_path):
        return False
    with zipfile.ZipFile(zip_path) as zf:
        return zf.testzip() is None

if VERIFIED_MARKER.exists() and VERIFIED_MARKER.read_text().strip() == str(ZIP_PATH.stat().st_size):
    print("Already verified (size unchanged).")
else:
    print("Running integrity check (full read pass)...")
    if check_integrity(ZIP_PATH):
        VERIFIED_MARKER.write_text(str(ZIP_PATH.stat().st_size))
        print("Integrity check passed.")
    else:
        ZIP_PATH.unlink()
        VERIFIED_MARKER.unlink(missing_ok=True)
        raise RuntimeError("Integrity check FAILED -- corrupted/incomplete zip deleted. Re-run the download cell.")


## 3. Extraction

In [ ]:

if EXTRACT_DIR.exists() and any(EXTRACT_DIR.rglob("*")):
    print("Already extracted:", EXTRACT_DIR)
else:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        for member in tqdm(zf.infolist(), desc="Extracting"):
            zf.extract(member, EXTRACT_DIR)
    print("Extraction complete:", EXTRACT_DIR)


## 4. Sanity check

In [ ]:

all_files = [p for p in EXTRACT_DIR.rglob("*") if p.is_file()]
video_files = [p for p in all_files if p.suffix.lower() in (".mp4", ".avi", ".mov", ".mkv")]
total_bytes = sum(p.stat().st_size for p in all_files)
print(f"Total files extracted: {len(all_files)}")
print(f"Video files: {len(video_files)}")
print(f"Total size on disk: {total_bytes / 1e9:.1f} GB")
print("Sample paths:", [str(p.relative_to(EXTRACT_DIR)) for p in all_files[:10]])


## Notes / caveats

- **If you hit the same 403 on Colab**: this points to an IP-based block
  rather than a code problem (the build environment hit the identical 403
  from a completely different network). The practical workaround is
  downloading once from a real browser on a normal residential/campus
  connection and uploading the result into
  `MyDrive/endofm-benchmark/data/pitvis/` yourself, matching the paths this
  notebook expects (`pitvis_all_files_v2.zip`, or extracted directly into
  `data/pitvis/raw/`).
- **Why PitVis-2023**: AutoLaparo's access link is dead; Kvasir-Capsule and
  Cholec80 (and CholecT50/CholecSeg8k/CholecTrack20/Endoscapes2023, all
  derived from the same CAMMA cholecystectomy video collections) are either
  weak signal or overlap with Endo-FM's own pretraining data. PitVis-2023's
  pituitary-neurosurgery domain has no plausible overlap with Endo-FM's
  pretraining corpus, and it's the actual target domain named in
  `neurosurgery_notegen_architecture.md`.
- This notebook only fetches raw data -- building the step/instrument
  linear-probe pipeline on top of it is a follow-up step.
